# OpenGrok Search API

Endpoint: https://apiuat-intra.wanhai.com/qad/services/opengrok/api/v1

Source: https://github.com/richard-1933/opengrok-search

### [Code Scan-1] 透過java呼叫的sp/sf

In [ ]:

import re, pandas as pd
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_spsf = [
    ('full', '/.*(sp|sf)\_[a-zA-Z]{3}\_.*/ -type:sh -type:jar -type:javaclass +path:/[a-zA-Z]{3}\_MP/'),
    ('path', '-". prc" -". fnc" -". trg" -". tri" -". viw" -". tab" -". pck" -". tps" -/develop -/branch -/IAL -/CNWEB')
]
# param_spsf.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))


records = codescan.scan(param_spsf, fetch_all=True, url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                        headers={"apikey": os.getenv('OPENGROK_API_KEY')})
if records:
    data = []
    for file_path, entries in records.items():
        system = re.findall(r"^/([^/]+)/", file_path, re.I)[0]
        dbobjs = set()
        for entry in entries:
            dbobjs.update(re.findall(r'<b>(.[sp|sf]_[a-zA-Z]{3}_[a-zA-Z]*[0-9]*)</b>', entry['line'], flags=re.I))

        for dbobj in dbobjs:
            data.append({
                "system": system,
                "dbobj": dbobj.upper()
            })
    df = pd.DataFrame(data)
    print(f"Ttl Scan Results:　{len(records)}, Ttl Output Count: {len(data)}")
    export_file = "c:\\temp\\output.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
    codescan.write_to_excel(df, export_file, 'system')
else:
    print("No records found.")
df


### [Code Scan-2] 內含row_id / rowid關鍵字的程式 (java)

In [ ]:
from opengrok_util import codescan
import os, platform
from dotenv import load_dotenv

load_dotenv(override=True)

param_rowid_java = [
    ('full', '"row_id" OR "rowid" type:java')
    # ('path', '')
]
# param_rowid_java.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))
export_file = "c:\\temp\\output.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
file_path, df = codescan.scan_to_excel(param_rowid_java, fetch_all=False, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### [Code Scan-3] 內含row_id / rowid關鍵字的程式 (db objs、shell.....)

In [ ]:
from opengrok_util import codescan
import os, platform
from dotenv import load_dotenv

load_dotenv(override=True)

param_row_id_other = [
    ('full', '"row_id" OR "rowid" -type:java -type:jar -type:javaclass')
    # ('path', '')
]
# param_row_id_other.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))
export_file = "c:\\temp\\output.xlsx" if platform.system() == 'Windows' else None
fiile_path, df = codescan.scan_to_excel(param_row_id_other, fetch_all=False, export_file=export_file,
                                        url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                        headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### [Code Scan-4] 內含ftp關鍵字的程式

In [ ]:
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_ftp_qad = [
    ('full', '("ftp" OR "ftputil") -type:jar -type:javaclass'),
    ('path', '-". js" -". htm" -/Production -/Beta -/Branch')
]
param_ftp_qad.append(('projects', [proj for proj in
                                   ['BKG', 'CCM', 'CMR', 'CRS', 'CSS', 'DDS', 'DGS', 'ECS', 'EDI', 'IHD', 'LMR', 'OOC',
                                    'PAM', 'QAD', 'SAS', 'SRS', 'SKD', 'SSM', 'WAS', 'WHL', 'WCS']]))

export_file = "c:\\temp\\output.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
file_path, df = codescan.scan_to_excel(param_ftp_qad, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### [Code Scan-5] 原始碼整理

In [ ]:
from opengrok_util import codescan
from dotenv import load_dotenv
import os, platform

load_dotenv(override=True)

param_src = [
    ('full', '-type:jar -type:javaclass'),
    ('path', '/build /classes')
]
# param_src.append(('projects', [proj for proj in ['BKG','CCM','CMR','CRS','CSS','DDS','DGS','ECS','EDI','IHD','LMR','OOC','PAM','QAD','SAS','SRS','SKD','SSM','WAS','WHL','WCS']]))

export_file = "c:\\temp\\output.xlsx" if platform.system() == 'Windows' else os.path.curdir + os.sep + 'scan_result.xlsx'
file_path, df = codescan.scan_to_excel(param_src, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df

### [Code Scan-6] DB-Link Usages

- 先由whl2、whi2、wip2找出所有指向SZP5.GLB2的synonym，再透過codescan找出所有內容含有這些關鍵字的程式

In [ ]:
import os, platform, pandas as pd
from dotenv import load_dotenv
from opengrok_util import codescan

load_dotenv(override=True)

# dba提供的glb2 dblink reference objs
excel_path = "C:\\Users\\mxxxx\\IdeaProjects\\python\\opengrok-search\\resources\\glb2_dblink.xlsx"

df = pd.read_excel(excel_path, sheet_name=None)

# df_whl2 = df['whl2'][['TYPE', 'OWNER', 'NAME', 'REFERENCED_LINK_NAME']]

# df_obj = pd.merge(df['whl2']['NAME'], df['whi2']['NAME'], on='NAME', how='outer')
# df_obj = pd.merge(df_obj['NAME'], df['wip2']['NAME'], on='NAME', how='outer')
df_obj = df['glb2']

param_src = [
    ('full', "-type:jar -type:javaclass (" + " OR ".join([obj_name for obj_name in df_obj['NAME'].unique()]) + ")")
]

export_file = "c:\\temp\\scan-result_dblink_from_glb2.xlsx" if platform.system() == 'Windows' else None
file_path, df = codescan.scan_to_excel(param_src, fetch_all=True, export_file=export_file,
                                       url=os.getenv('OPENGROK_API_ENDPOINT') + "/search",
                                       headers={"apikey": os.getenv('OPENGROK_API_KEY')})
df


### [Code Scan-7] DB-Link Usages (CLD5)

- 先由cld5找出所有dblink referenced objs，再透過opengrok進行opengrok比對

In [ ]:
import os, platform,cx_Oracle, pandas as pd
from dotenv import load_dotenv
from opengrok_util import codescan

load_dotenv(override=True)

db_url = os.getenv("CLD5_DB_URL")
db_port = os.getenv("CLD5_DB_PORT")
db_sid = os.getenv("CLD5_DB_SID")
db_user = os.getenv("CLD5_DB_USER")
db_pwd = os.getenv("CLD5_DB_PWD")

# 設置 Oracle Instant Client 的位置
oracle_client_path = r'C:\instantclient_19_25'  # 替換為實際的 Oracle Instant Client 路徑
os.environ['PATH'] = oracle_client_path + os.pathsep + os.environ['PATH']

# Define the connection parameters
dsn_tns = cx_Oracle.makedsn(db_url, db_port, sid=db_sid)
connection = cx_Oracle.connect(user=db_user, password=db_pwd, dsn=dsn_tns)

query="""
    select type,owner,  name,REFERENCED_LINK_NAME from dba_dependencies
    where REFERENCED_LINK_NAME is not null
    group by name,type,owner , referenced_link_name
    union all
    select 'SYNONYM' as type,owner,synonym_name as name,db_link as REFERENCED_LINK_NAME
    from dba_synonyms
    where
        db_link is not null
        -- db_link like upper(${referenced_link_name})
    group by 'SYNONYM',owner,synonym_name,db_link
    order by 4,3
"""

try:
    df = pd.read_sql(query, con=connection)
finally:
    connection.close()
df


### [Sample] Read Data from Oracle

In [ ]:
import pandas as pd
import os
import cx_Oracle
from dotenv import load_dotenv

load_dotenv(override=True)

db_url = os.getenv("CLD5_DB_URL")
db_port = os.getenv("CLD5_DB_PORT")
db_sid = os.getenv("CLD5_DB_SID")
db_user = os.getenv("CLD5_DB_USER")
db_pwd = os.getenv("CLD5_DB_PWD")

# 設置 Oracle Instant Client 的位置
oracle_client_path = r'C:\instantclient_19_25'  # 替換為實際的 Oracle Instant Client 路徑
os.environ['PATH'] = oracle_client_path + os.pathsep + os.environ['PATH']

# Define the connection parameters
dsn_tns = cx_Oracle.makedsn(db_url, db_port, sid=db_sid)
connection = cx_Oracle.connect(user=db_user, password=db_pwd, dsn=dsn_tns)

query=os.getenv("SQLSTMT_GET_DBLINK")

df = pd.read_sql(query, con=connection)

connection.close()

df
